# Baseline Model Training from Snapshot (API, Max Opt)

This notebook builds or reuses a taiko-only constant-BPM snapshot and then trains only the baseline model using the direct Python API.

Flow:
1. Build or reuse a 1000-set snapshot under `C:/taiko-transformer-cache/snapshots`.
2. Create or resume the baseline training context from the snapshot dataset.
3. Train the baseline model and inspect the resulting checkpoint paths.


In [1]:
from pathlib import Path

import torch

from src.model import (
    ArchitectureSpec,
    TrainingSpec,
    WandbConfig,
    build_training_artifacts,
    create_training_context,
    load_training_context_from_checkpoint,
    train_context,
)
from src.preprocessing import build_snapshot_dataset

repo_root = Path.cwd()
source_unpacked_root = Path("C:/taiko-transformer-cache/unpacked")
snapshot_target_set_count = 1000
snapshot_seed = 42
max_audio_mb = 5.0
snapshot_root = Path(
    f"C:/taiko-transformer-cache/snapshots/taiko_only_static_bpm_{snapshot_target_set_count}_seed{snapshot_seed}"
)
data_root = snapshot_root
training_dir = data_root / "training"
checkpoints_dir = repo_root / "checkpoints" / "baseline_snapshot_maxopt"
last_checkpoint = checkpoints_dir / "last.ckpt"
best_checkpoint = checkpoints_dir / "best.ckpt"

index_cache_dir = training_dir / "index_cache"
inference_snapshots_dir = checkpoints_dir / "inference_snapshots"

epochs = 10
batch_size = 16
num_workers = 0
precision = "auto"
pin_memory = True
persistent_workers = False
prefetch_factor = 2
architecture_name = "taiko_transformer"
keep_only_max_notes_per_song = True
build_snapshot = False
overwrite_snapshot = False
save_inference_every_n_steps = 1000
run_name = "baseline_snapshot_maxopt"
use_resume_if_available = True
use_wandb = False
wandb_log_every_batches = 100
wandb_notebook_name = "train_baseline_snapshot_api_maxopt.ipynb"
wandb_api_key = ""
wandb_offline = False

if torch.cuda.is_available():
    best_device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    best_device = "mps"
else:
    best_device = "cpu"

checkpoints_dir.mkdir(parents=True, exist_ok=True)

print(f"repo_root                     : {repo_root}")
print(f"source_unpacked_root          : {source_unpacked_root}")
print(f"snapshot_root                 : {snapshot_root}")
print(f"data_root                     : {data_root}")
print(f"training_dir                  : {training_dir}")
print(f"checkpoints_dir               : {checkpoints_dir}")
print(f"last_checkpoint               : {last_checkpoint}")
print(f"index_cache_dir               : {index_cache_dir}")
print(f"inference_snapshots_dir       : {inference_snapshots_dir}")
print(f"best_checkpoint               : {best_checkpoint}")
print(f"snapshot_target_set_count     : {snapshot_target_set_count}")
print(f"keep_only_max_notes_per_song  : {keep_only_max_notes_per_song}")
print(f"best_device                   : {best_device}")


c:\Users\28548\.conda\envs\pytorch\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\28548\.conda\envs\pytorch\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


repo_root                     : c:\Users\28548\PythonNotebooks\taiko-diffusion
source_unpacked_root          : C:\taiko-transformer-cache\unpacked
snapshot_root                 : C:\taiko-transformer-cache\snapshots\taiko_only_static_bpm_1000_seed42
data_root                     : C:\taiko-transformer-cache\snapshots\taiko_only_static_bpm_1000_seed42
training_dir                  : C:\taiko-transformer-cache\snapshots\taiko_only_static_bpm_1000_seed42\training
checkpoints_dir               : c:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\baseline_snapshot_maxopt
last_checkpoint               : c:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\baseline_snapshot_maxopt\last.ckpt
index_cache_dir               : C:\taiko-transformer-cache\snapshots\taiko_only_static_bpm_1000_seed42\training\index_cache
inference_snapshots_dir       : c:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\baseline_snapshot_maxopt\inference_snapshots
best_checkpoint               : c

## Step 1: Build or reuse the snapshot dataset


In [2]:
snapshot_summary = None
required_snapshot_files = [
    snapshot_root / "chart_index" / "chart_build_summary.csv",
    snapshot_root / "beat_aligned_dataset" / "sequence_metadata.csv",
]
snapshot_ready = all(path.exists() for path in required_snapshot_files)
should_build_snapshot = build_snapshot or not snapshot_ready

if should_build_snapshot:
    if not snapshot_ready and not build_snapshot:
        print("Snapshot dataset artifacts were not found; building the snapshot automatically.")
    elif build_snapshot:
        print("Building snapshot dataset because build_snapshot=True.")
    snapshot_summary = build_snapshot_dataset(
        source_unpacked_root=source_unpacked_root,
        snapshot_root=snapshot_root,
        target_set_count=snapshot_target_set_count,
        seed=snapshot_seed,
        max_audio_mb=max_audio_mb,
        overwrite=overwrite_snapshot,
        keep_only_max_notes_per_song=keep_only_max_notes_per_song,
    )
    print(snapshot_summary)
else:
    print("Skipping snapshot build and reusing existing snapshot dataset.")

baseline_artifacts = build_training_artifacts(data_root, checkpoints_dir=checkpoints_dir)
print(baseline_artifacts)


Snapshot dataset artifacts were not found; building the snapshot automatically.


c:\Users\28548\.conda\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\28548\.conda\envs\pytorch\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


Scanned set count   : 10025
Eligible set count  : 4047
Selected set count  : 1000
Rejection breakdown : audio_count_error=125, audio_too_large=3114, non_constant_bpm=766, non_taiko_mode=1973
Snapshot root       : C:\taiko-transformer-cache\snapshots\taiko_only_static_bpm_1000_seed42
{'source_unpacked_root': 'C:\\taiko-transformer-cache\\unpacked', 'snapshot_root': 'C:\\taiko-transformer-cache\\snapshots\\taiko_only_static_bpm_1000_seed42', 'selection_manifest_csv': 'C:\\taiko-transformer-cache\\snapshots\\taiko_only_static_bpm_1000_seed42\\selection_manifest.csv', 'rejection_report_csv': 'C:\\taiko-transformer-cache\\snapshots\\taiko_only_static_bpm_1000_seed42\\rejection_report.csv', 'scanned_set_count': 10025, 'eligible_set_count': 4047, 'selected_set_count': 1000, 'rejection_breakdown': {'audio_count_error': 125, 'audio_too_large': 3114, 'non_constant_bpm': 766, 'non_taiko_mode': 1973}, 'target_set_count': 1000, 'seed': 42}
TrainingArtifacts(data_root=WindowsPath('C:/taiko-transform

## Step 2: Create or resume the baseline training context


In [3]:
baseline_architecture_spec = ArchitectureSpec(name=architecture_name)

baseline_training_spec = TrainingSpec(
    epochs=epochs,
    batch_size=batch_size,
    num_workers=num_workers,
    device=best_device,
    precision=precision,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
    prefetch_factor=prefetch_factor,
)

wandb_config = None
if use_wandb:
    wandb_config = WandbConfig(
        enabled=True,
        run_name=run_name,
        log_every_n_batches=wandb_log_every_batches,
        notebook_name=wandb_notebook_name,
        offline=wandb_offline,
        api_key=wandb_api_key,
        mode_name_for_run=architecture_name,
    )

if use_resume_if_available and last_checkpoint.exists():
    baseline_context = load_training_context_from_checkpoint(
        last_checkpoint,
        data_root=data_root,
        device=best_device,
        batch_size=batch_size,
        num_workers=num_workers,
        checkpoints_dir=checkpoints_dir,
        index_cache_dir=index_cache_dir,
        precision=precision,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
    )
    print(f"Resuming from checkpoint: {last_checkpoint}")
else:
    baseline_context = create_training_context(
        data_root=data_root,
        architecture_spec=baseline_architecture_spec,
        training_spec=baseline_training_spec,
        checkpoints_dir=checkpoints_dir,
        index_cache_dir=index_cache_dir,
        precision=precision,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
    )
    print("Starting a fresh baseline-model snapshot training run.")

print("baseline architecture:", baseline_context.architecture_spec)
print("baseline ignore_index:", baseline_context.dataset.label_ignore_index)
print("start_epoch:", baseline_context.start_epoch)
print("target_epochs:", epochs)


[startup] index cache lookup...
[startup] index cache lookup done in 0.00s
[startup] manifest...
[startup] manifest done in 38.08s
[startup] splits...
[startup] splits done in 0.00s
[startup] indexes...
[startup] indexes done in 0.91s
[startup] index cache save...
[startup] index cache save done in 0.05s
[startup] vocab...
[startup] vocab done in 0.55s
[startup] dataset objects...
[startup] dataset objects done in 0.00s
Starting a fresh baseline-model snapshot training run.
baseline architecture: ArchitectureSpec(name='taiko_transformer', input_dim=128, d_model=256, nhead=4, num_encoder_layers=4, num_decoder_layers=4, dim_feedforward=1024, dropout=0.1, max_len=512, history_max_tokens=256, retrieval_top_k=1, retrieval_max_tokens_per_window=24, retrieval_exclude_last_n_windows=2, use_motif_retrieval=True, max_cached_charts=4)
baseline ignore_index: 0
start_epoch: 1
target_epochs: 10


## Step 3: Train the baseline model


In [4]:
baseline_context = train_context(
    baseline_context,
    epochs=epochs,
    log_every_n_batches=wandb_log_every_batches,
    wandb_config=wandb_config,
    save_inference_every_n_steps=save_inference_every_n_steps,
    inference_snapshots_dir=inference_snapshots_dir,
)

print("Training finished.")
print(f"last checkpoint: {last_checkpoint.resolve()}")
print(f"best checkpoint: {best_checkpoint.resolve()}")
print(f"inference snapshots dir: {inference_snapshots_dir.resolve()}")


[runtime] precision requested=auto resolved=bf16 autocast=1 scaler=0
[runtime] inference_snapshots every_n_steps=1000 dir=C:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\baseline_snapshot_maxopt\inference_snapshots


Training:   0%|          | 0/4097 [00:00<?, ?it/s]c:\Users\28548\.conda\envs\pytorch\Lib\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)


Epoch 1/10 | lr: 0.000100 | train loss: 0.9071 | val loss: 0.8454 | train samp/s: 213.26 | train tok/s: 6971.22


Epoch 2/10 | lr: 0.000100 | train loss: 0.8171 | val loss: 0.8459 | train samp/s: 173.26 | train tok/s: 5672.43


Epoch 3/10 | lr: 0.000100 | train loss: 0.7949 | val loss: 0.8148 | train samp/s: 142.44 | train tok/s: 4660.31


...

Epoch 6/10 | lr: 0.000100 | train loss: 0.7635 | val loss: 0.8115 | train samp/s: 158.28 | train tok/s: 5177.16


Epoch 7/10 | lr: 0.000100 | train loss: 0.7560 | val loss: 0.8090 | train samp/s: 140.60 | train tok/s: 4596.90


Epoch 8/10 | lr: 0.000100 | train loss: 0.7480 | val loss: 0.8076 | train samp/s: 136.84 | train tok/s: 4487.55


Epoch 9/10 | lr: 0.000100 | train loss: 0.7415 | val loss: 0.8054 | train samp/s: 136.53 | train tok/s: 4470.65


Epoch 10/10 | lr: 0.000100 | train loss: 0.7395 | val loss: 0.8165 | train samp/s: 136.47 | train tok/s: 4471.36
Training finished.
last checkpoint: C:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\baseline_snapshot_maxopt\last.ckpt
best checkpoint: C:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\baseline_snapshot_maxopt\best.ckpt
inference snapshots dir: C:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\baseline_snapshot_maxopt\inference_snapshots


## Optional inspection


In [5]:
print("history keys:", baseline_context.history.keys())
print("snapshot root:", snapshot_root)
print("training dir:", training_dir)
print("selection manifest:", snapshot_root / "selection_manifest.csv")
print("rejection report:", snapshot_root / "rejection_report.csv")
print("vocab json:", training_dir / "vocab.json")
print("splits json:", training_dir / "splits.json")


history keys: dict_keys(['train_loss', 'val_loss', 'lr', 'train_density_proxy_abs_error', 'val_density_proxy_abs_error', 'train_difficulty_proxy_drift', 'val_difficulty_proxy_drift'])
snapshot root: C:\taiko-transformer-cache\snapshots\taiko_only_static_bpm_1000_seed42
training dir: C:\taiko-transformer-cache\snapshots\taiko_only_static_bpm_1000_seed42\training
selection manifest: C:\taiko-transformer-cache\snapshots\taiko_only_static_bpm_1000_seed42\selection_manifest.csv
rejection report: C:\taiko-transformer-cache\snapshots\taiko_only_static_bpm_1000_seed42\rejection_report.csv
vocab json: C:\taiko-transformer-cache\snapshots\taiko_only_static_bpm_1000_seed42\training\vocab.json
splits json: C:\taiko-transformer-cache\snapshots\taiko_only_static_bpm_1000_seed42\training\splits.json
